# 02 - Data Cleaning

Clean `matches.csv`: missing values, duplicates, team name standardization, date parsing, unused columns, inconsistent categories.

In [1]:
import pandas as pd

matches = pd.read_csv('../data/raw/matches.csv')
matches = matches.rename(columns={'match_id': 'id'})
matches.shape

(1212, 19)

## 1. Missing values

In [2]:
matches.isnull().sum().sort_values(ascending=False)

city               51
winner             25
player_of_match     9
id                  0
season              0
team1_won           0
balls_per_over      0
overs               0
gender              0
match_type          0
event_name          0
result_type         0
date                0
toss_decision       0
toss_winner         0
team2               0
team1               0
venue               0
toss_winner_won     0
dtype: int64

In [3]:
# City is sometimes missing when venue name already encodes the city (e.g. Dubai matches).
# Fill from a venue -> city lookup built off known rows, then fall back to the text before
# the last comma in the venue name (venues are formatted "Ground, City").
venue_city = matches.dropna(subset=['city']).drop_duplicates('venue').set_index('venue')['city']
matches['city'] = matches['city'].fillna(matches['venue'].map(venue_city))
matches['city'] = matches['city'].fillna(matches['venue'].str.rsplit(',', n=1).str[-1].str.strip())

# player_of_match is only missing for no-result matches (nobody could be named), not bad data.
matches['player_of_match'] = matches['player_of_match'].fillna('No Result')

## 2. Duplicates

In [4]:
print('duplicate rows:', matches.duplicated().sum())
matches = matches.drop_duplicates()

duplicate rows: 0


## 3. Standardize team names

Franchises that were rebranded or relocated get mapped to a single current name so win-rate stats aren't split across eras.

In [5]:
TEAM_NAME_MAP = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Pune Warriors': 'Pune Warriors India',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}

team_cols = ['team1', 'team2', 'toss_winner', 'winner']
for col in team_cols:
    matches[col] = matches[col].replace(TEAM_NAME_MAP)

## 4. Date columns

In [6]:
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

## 5. Drop columns not useful for prediction

`event_name`, `match_type`, and `gender` are constant across the dataset (always IPL, T20, male). `overs`/`balls_per_over` are likewise constant (20, 6). `result_type` just lists which raw fields were populated upstream, not a usable category. `team1_won`/`toss_winner_won` are the source's own precomputed labels — recomputed independently in `05_preprocessing.ipynb` rather than trusted as-is.

In [7]:
DROP_COLS = ['event_name', 'match_type', 'gender', 'overs', 'balls_per_over',
             'result_type', 'team1_won', 'toss_winner_won']
matches = matches.drop(columns=[c for c in DROP_COLS if c in matches.columns])

## 6. Inconsistent categorical values

`season` is stored as a mix of formats (`2024` vs `2007/08`) — normalized to the starting year as an integer. `venue` is stored inconsistently — the same ground appears as `"Wankhede Stadium"`, `"Wankhede Stadium, Mumbai"`, sometimes even `"..., Mohali, Chandigarh"` — a mix of `"Ground"` and `"Ground, City[, City2]"`. Since `city` is already its own clean column, `venue` is canonicalized to just the ground name (everything before the first comma), which collapses 59 raw venue strings down to 42 real venues.

In [8]:
matches['city'] = matches['city'].str.strip()
matches['venue'] = matches['venue'].str.split(',').str[0].str.strip()
matches['toss_decision'] = matches['toss_decision'].str.lower().str.strip()
matches['season'] = matches['season'].astype(str).str.split('/').str[0].astype(int)

## Save cleaned dataset

In [9]:
matches.to_csv('../data/processed/matches_cleaned.csv', index=False)
matches.shape

(1212, 11)